# Training Data Infrastructure

Hands-on comparison of the two dominant training data formats for large-scale ML: **WebDataset** (tar-based) and **MosaicML Streaming** (MDS). Both solve the same problem — serving data to distributed training faster than naive file I/O — but make different tradeoffs.

**VRAM:** CPU-only (data loading benchmarks). **Time:** ~2–3 hours.

**Dependencies:** `webdataset`, `mosaicml-streaming`

## Why Specialized Data Formats?

Naive approach: millions of individual image/video files on disk or cloud storage.

**Problems at scale:**
- **File system overhead:** Opening millions of small files is slow (inode lookups, metadata reads)
- **Random I/O:** SSDs handle random reads well, but cloud storage (S3) and HDDs do not
- **Network bottleneck:** Fetching individual files from S3 = one HTTP request per sample
- **Shuffle quality:** True random shuffle across a billion samples requires loading the full index into memory
- **Multi-node consistency:** N workers on M nodes need non-overlapping, balanced data splits

**Solution:** Pack samples into larger sequential files (tar shards or binary chunks) and read them sequentially.

| Property | Individual Files | WebDataset (tar) | MDS (binary chunks) |
|---|---|---|---|
| File count (1M samples) | 1M+ files | ~1000 tar files | ~1000 MDS files |
| Sequential read speed | Poor | Excellent | Excellent |
| S3 streaming | 1 req/sample | 1 req/shard | 1 req/chunk |
| Shuffle quality | Perfect (if RAM) | Within-shard + shuffle buffer | Deterministic global |
| Resume from checkpoint | Trivial | Needs shard tracking | Built-in |
| Multi-node splitting | Manual | Shard-based | Built-in |

### Why Not Parquet?

Parquet is the default columnar format for analytics and tabular ML. But it breaks down for **multimodal foundation model training**:

| Problem | Why It Hurts |
|---|---|
| **No fast random access** | Parquet is optimized for sequential scans, not random row lookups. Training with global shuffling requires O(1) access to any row — Parquet needs to scan row groups. |
| **Extreme columnar variance** | A training table has scalar columns (quality scores), fixed-size arrays (embeddings), and massive blobs (compressed video). Parquet's row-group layout wastes space and I/O when columns vary 1000x in size. |
| **Copy-per-experiment is impossible** | At petabyte scale, you cannot materialize a new dataset per training run. You need to query in-place with predicate pushdown and column selection. Partitioned Parquet makes this slow. |
| **Schema evolution requires rewriting** | Swapping your embedding model (CLIP → SigLIP) means adding a new column. Parquet requires rewriting entire files. Lance supports append-only column addition. |
| **No native vector index** | Embedding search requires a separate FAISS/Milvus service alongside Parquet. Lance has built-in IVF_PQ indexing — one fewer system to maintain. |

**Lance** solves these: 100x faster random access, columnar with native vector index, append-only schema evolution, zero-copy PyTorch integration. This is why leading video generation labs use Lance as their canonical data store, with WebDataset/MDS as downstream training serving formats.

The rest of this notebook focuses on WebDataset and MDS — the formats you'll use to **serve** training data. But understand that they sit downstream of a feature lakehouse (covered in [02 — Video Data Pipeline](02_video_data_pipeline.ipynb)).

## Part 1: WebDataset

[WebDataset](https://github.com/webdataset/webdataset) stores samples as consecutive files within tar archives. Each sample is a group of files sharing the same key (e.g., `000042.jpg`, `000042.txt`, `000042.json`).

In [ ]:
import os
import time
import json
import tempfile
from pathlib import Path

import torch
from torch.utils.data import DataLoader
import webdataset as wds
import numpy as np
from PIL import Image

### Writing WebDataset Shards

`ShardWriter` packs samples into tar files with a configurable max size per shard.

In [ ]:
def create_synthetic_dataset(output_dir: Path, n_samples: int = 5000, image_size: int = 256) -> Path:
    """Create a synthetic image+caption dataset in WebDataset format."""
    output_dir.mkdir(parents=True, exist_ok=True)
    shard_pattern = str(output_dir / "shard-%05d.tar")

    with wds.ShardWriter(shard_pattern, maxcount=1000) as sink:
        for i in range(n_samples):
            # Synthetic image: random noise with a colored square
            img = np.random.randint(0, 50, (image_size, image_size, 3), dtype=np.uint8)
            cx, cy = np.random.randint(50, image_size - 50, 2)
            color = np.random.randint(100, 255, 3)
            img[cy-25:cy+25, cx-25:cx+25] = color

            caption = f"A {['red','green','blue','yellow','purple'][i % 5]} square on dark background, sample {i}"
            metadata = {"width": image_size, "height": image_size, "source": "synthetic", "index": i}

            sink.write({
                "__key__": f"sample{i:06d}",
                "jpg": Image.fromarray(img),
                "txt": caption,
                "json": metadata,
            })

    n_shards = len(list(output_dir.glob("shard-*.tar")))
    print(f"Wrote {n_samples} samples across {n_shards} shards to {output_dir}")
    return output_dir

# Create dataset
wds_dir = Path(tempfile.mkdtemp()) / "wds_shards"
create_synthetic_dataset(wds_dir, n_samples=5000)

### Reading WebDataset Shards

The reading pipeline chains operations: open shards → decode → shuffle → batch.

In [ ]:
def create_wds_dataloader(
    shard_dir: Path,
    batch_size: int = 32,
    num_workers: int = 4,
    shuffle_buffer: int = 1000,
) -> DataLoader:
    """Create a WebDataset DataLoader with shuffle buffer."""
    shard_urls = sorted(str(p) for p in shard_dir.glob("shard-*.tar"))
    url_string = shard_urls  # wds accepts a list of paths

    dataset = (
        wds.WebDataset(url_string)
        .shuffle(shuffle_buffer)
        .decode("pil")
        .to_tuple("jpg", "txt", "json")
        .map_tuple(
            lambda img: torch.tensor(np.array(img)).permute(2, 0, 1).float() / 255.0,  # HWC→CHW, normalize
            lambda txt: txt,
            lambda meta: meta,
        )
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
    )

# Test read
loader = create_wds_dataloader(wds_dir, batch_size=32, num_workers=2)
batch = next(iter(loader))
images, captions, metadata = batch
print(f"Batch shape: {images.shape}")
print(f"First caption: {captions[0]}")
print(f"First metadata: {metadata[0]}")

### Throughput Measurement

In [ ]:
def measure_throughput(loader, n_batches: int = 50, warmup: int = 5) -> dict:
    """Measure data loading throughput (samples/sec)."""
    total_samples = 0
    it = iter(loader)

    # Warmup
    for _ in range(warmup):
        try:
            batch = next(it)
        except StopIteration:
            it = iter(loader)
            batch = next(it)

    # Measure
    start = time.perf_counter()
    for _ in range(n_batches):
        try:
            batch = next(it)
        except StopIteration:
            break
        total_samples += batch[0].shape[0]
    elapsed = time.perf_counter() - start

    return {
        "total_samples": total_samples,
        "elapsed_sec": round(elapsed, 2),
        "samples_per_sec": round(total_samples / elapsed, 1),
        "batches_measured": n_batches,
    }

wds_stats = measure_throughput(loader, n_batches=50)
print(f"WebDataset throughput: {wds_stats['samples_per_sec']} samples/sec")
print(f"  ({wds_stats['total_samples']} samples in {wds_stats['elapsed_sec']}s)")

## Part 2: MosaicML Streaming (MDS)

[MosaicML Streaming](https://github.com/mosaicml/streaming) writes samples to binary MDS files with a global index. Key advantage: **deterministic shuffling** — given the same seed, every worker on every node sees the same global order.

In [ ]:
from streaming import MDSWriter, StreamingDataset

### Writing MDS Format

In [ ]:
def create_mds_dataset(output_dir: Path, n_samples: int = 5000, image_size: int = 256) -> Path:
    """Create the same synthetic dataset in MDS format."""
    output_dir.mkdir(parents=True, exist_ok=True)

    columns = {
        "image": "jpeg",
        "caption": "str",
        "metadata": "json",
    }

    with MDSWriter(out=str(output_dir), columns=columns, size_limit=64 * 1024 * 1024) as writer:
        for i in range(n_samples):
            img = np.random.randint(0, 50, (image_size, image_size, 3), dtype=np.uint8)
            cx, cy = np.random.randint(50, image_size - 50, 2)
            color = np.random.randint(100, 255, 3)
            img[cy-25:cy+25, cx-25:cx+25] = color

            pil_img = Image.fromarray(img)
            caption = f"A {['red','green','blue','yellow','purple'][i % 5]} square on dark background, sample {i}"
            metadata = {"width": image_size, "height": image_size, "source": "synthetic", "index": i}

            writer.write({
                "image": pil_img,
                "caption": caption,
                "metadata": metadata,
            })

    n_files = len(list(output_dir.glob("*.mds")))
    print(f"Wrote {n_samples} samples across {n_files} MDS files to {output_dir}")
    return output_dir

mds_dir = Path(tempfile.mkdtemp()) / "mds_shards"
create_mds_dataset(mds_dir, n_samples=5000)

### Reading MDS Format

In [ ]:
def create_mds_dataloader(
    data_dir: Path,
    batch_size: int = 32,
    num_workers: int = 4,
) -> DataLoader:
    """Create a Streaming DataLoader."""
    local_dir = Path(tempfile.mkdtemp()) / "mds_local"

    dataset = StreamingDataset(
        local=str(data_dir),
        shuffle=True,
        batch_size=batch_size,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
    )

mds_loader = create_mds_dataloader(mds_dir, batch_size=32, num_workers=2)
sample = next(iter(mds_loader))
print(f"Keys: {list(sample.keys())}")
print(f"Caption: {sample['caption'][0]}")

In [ ]:
def measure_mds_throughput(loader, n_batches: int = 50, warmup: int = 5) -> dict:
    """Measure MDS loading throughput."""
    total_samples = 0
    it = iter(loader)

    for _ in range(warmup):
        try:
            batch = next(it)
        except StopIteration:
            it = iter(loader)
            batch = next(it)

    start = time.perf_counter()
    for _ in range(n_batches):
        try:
            batch = next(it)
        except StopIteration:
            break
        total_samples += len(batch['caption'])
    elapsed = time.perf_counter() - start

    return {
        "total_samples": total_samples,
        "elapsed_sec": round(elapsed, 2),
        "samples_per_sec": round(total_samples / elapsed, 1),
    }

mds_stats = measure_mds_throughput(mds_loader, n_batches=50)
print(f"MDS throughput: {mds_stats['samples_per_sec']} samples/sec")
print(f"  ({mds_stats['total_samples']} samples in {mds_stats['elapsed_sec']}s)")

## Part 3: Side-by-Side Comparison

In [ ]:
print("=" * 60)
print("THROUGHPUT COMPARISON (same dataset, same hardware)")
print("=" * 60)
print(f"{'Format':<20} {'Samples/sec':>15} {'Time (s)':>10}")
print("-" * 60)
print(f"{'WebDataset':<20} {wds_stats['samples_per_sec']:>15.1f} {wds_stats['elapsed_sec']:>10.2f}")
print(f"{'MDS':<20} {mds_stats['samples_per_sec']:>15.1f} {mds_stats['elapsed_sec']:>10.2f}")
print("=" * 60)
print()
print("Note: Throughput varies with hardware, num_workers, and dataset size.")
print("Run on your target hardware for meaningful comparisons.")

### Decision Matrix

| Factor | WebDataset | MDS (Streaming) | Winner |
|---|---|---|---|
| **Ecosystem maturity** | Longer history, widely adopted | Newer, growing fast | WebDataset |
| **Shuffle quality** | Shuffle buffer (approximate) | Deterministic global shuffle | MDS |
| **Checkpoint resume** | Manual shard tracking | Built-in, exact sample resume | MDS |
| **Multi-node splitting** | Shard-based (uneven if shard sizes vary) | Balanced by design | MDS |
| **S3/cloud streaming** | Native (URL-based) | Native (remote=URL) | Tie |
| **Format simplicity** | tar files (inspect with `tar -tf`) | Binary (needs MDS tools) | WebDataset |
| **Video support** | Straightforward (video as bytes in tar) | Supported but less common | WebDataset |
| **Adoption in video gen** | Very common (Open-Sora, SVD) | Growing (MosaicML users) | WebDataset |

**Recommendation:** For video generation training data, **WebDataset is the safer default** — it's the de facto standard in the video gen community. Use MDS when you need deterministic shuffle or exact checkpoint resume (especially for very long training runs where reproducibility matters).

## Part 4: Distributed Data Loading

### How Distribution Works

In distributed training (DDP/FSDP), each GPU process needs a **non-overlapping subset** of the data:

```
Global Dataset (N samples)
    ↓
┌──────────────┬──────────────┬──────────────┬──────────────┐
│   Rank 0     │   Rank 1     │   Rank 2     │   Rank 3     │
│  N/4 samples │  N/4 samples │  N/4 samples │  N/4 samples │
└──────────────┴──────────────┴──────────────┴──────────────┘
```

**PyTorch DistributedSampler:** Splits indices across ranks. Works with any Dataset but:
- Requires the full index in memory (fine for millions, painful for billions)
- Padding to make all ranks equal length can bias the last batch
- No built-in checkpoint resume

**WebDataset distributed:** Splits *shards* (not samples) across ranks. Each rank reads different tar files.
- Pro: No index needed, streams from disk/S3
- Con: Unbalanced if shard sizes vary (mitigate: use consistent shard sizes)
- `wds.WebDataset(urls).shard_fn(split_by_worker)` for worker-level splitting

**MDS distributed:** Splits samples deterministically across ranks using a global shuffle algorithm.
- Pro: Perfectly balanced, deterministic, exact resume
- Con: Needs the global index file (small, but must be accessible to all ranks)
- `StreamingDataset` auto-detects `WORLD_SIZE` and `RANK` environment variables

In [ ]:
# Simulating distributed splitting (single-machine demo)

# WebDataset: shard-based splitting
shard_urls = sorted(str(p) for p in wds_dir.glob("shard-*.tar"))
n_shards = len(shard_urls)
world_size = 4  # simulating 4 GPUs

print("WebDataset shard assignment:")
for rank in range(world_size):
    assigned = shard_urls[rank::world_size]
    print(f"  Rank {rank}: {len(assigned)} shards — {[Path(s).name for s in assigned]}")

print()

# MDS: sample-based splitting (conceptual)
n_samples = 5000
print("MDS sample assignment (deterministic):")
for rank in range(world_size):
    start = rank * (n_samples // world_size)
    end = start + (n_samples // world_size)
    print(f"  Rank {rank}: samples [{start}:{end}] ({end - start} samples)")

### Diagnosing Data Loading Bottlenecks

| Symptom | Likely Cause | Fix |
|---|---|---|
| GPU utilization < 80% with high CPU | Data loading is the bottleneck | Increase `num_workers`, use faster storage |
| GPU utilization < 80% with low CPU | Data loading from slow disk/network | Move data to local NVMe, use larger shards |
| First epoch slow, subsequent fast | Data not cached locally | Pre-cache shards on local storage |
| Training loss spikes at shard boundaries | Poor shuffle quality | Increase shuffle buffer, use MDS deterministic shuffle |
| OOM on data loading workers | Workers loading too much into memory | Reduce `prefetch_factor`, reduce `num_workers` |
| Uneven GPU utilization across ranks | Unbalanced data splitting | Ensure equal shard sizes (WebDataset) or use MDS |

**Quick diagnosis recipe:**
```python
# Add to training loop:
import time
t0 = time.perf_counter()
batch = next(data_iter)
data_time = time.perf_counter() - t0

t0 = time.perf_counter()
loss = model(batch)
loss.backward()
compute_time = time.perf_counter() - t0

# If data_time > compute_time, you're data-bottlenecked
```

## Part 5: Ray Data and Orchestration at Scale

At production scale, the data loading patterns above (WebDataset, MDS) handle the **training side** — serving pre-built shards to distributed GPUs. But there's an equally important **preprocessing side**: the distributed compute that transforms raw data into those shards.

### Ray Data for Distributed Preprocessing

[Ray Data](https://docs.ray.io/en/latest/data/data.html) provides a distributed data processing API that's Python-native and GPU-aware:

```python
import ray

# Distributed video preprocessing with Ray Data
ds = ray.data.read_images("s3://raw-videos/frames/")
ds = (
    ds.map(resize_and_normalize)           # CPU: resize frames
    .map_batches(                           # GPU: run VLM captioning
        VLMCaptioner,
        compute=ray.data.ActorPoolStrategy(size=8),  # 8 GPU actors
        num_gpus=1,
    )
    .map(compute_quality_scores)            # CPU: aesthetic + motion scores
    .map_batches(                           # GPU: compute embeddings
        CLIPEmbedder,
        compute=ray.data.ActorPoolStrategy(size=4),
        num_gpus=1,
    )
)
ds.write_parquet("s3://processed/")        # or write to Lance
```

**Why Ray (vs. Spark, Dask, etc.):**
- **Heterogeneous compute:** Mix CPU and GPU tasks in the same pipeline. Spark can't schedule GPU actors.
- **Python-native:** No JVM overhead, no PySpark serialization. Your preprocessing code IS your Ray code.
- **Actor pools:** Long-lived GPU processes (VLM models stay loaded) instead of per-task model loading.
- **Integrates with training:** Same Ray cluster can run both preprocessing and distributed training.

### Ray Serve for Model Inference in Preprocessing

A common pattern: **separate preprocessing compute from training compute**.

- Training GPUs should be 100% utilized on gradient computation
- Preprocessing (VLM captioning, aesthetic scoring, embedding) runs on a **separate pool** via Ray Serve
- Ray Serve handles batching, autoscaling, and GPU memory management for inference models

```
Training Cluster (GPU)          Preprocessing Cluster (CPU + GPU)
┌──────────────────────┐       ┌──────────────────────────────┐
│ DDP/FSDP training    │       │ Ray Serve                    │
│ Reads from shards    │       │ ├── VLM Captioner (GPU)      │
│ 100% GPU utilization │       │ ├── CLIP Embedder (GPU)      │
│                      │       │ ├── Quality Scorer (CPU)     │
│                      │       │ └── Frame Decoder (CPU)      │
└──────────────────────┘       └──────────────────────────────┘
```

### Workflow Orchestration: Flyte, Airflow, Prefect

For the overall pipeline DAG (ingest → dedup → filter → caption → shard), you need a workflow orchestrator:

| Orchestrator | Strengths | Best For |
|---|---|---|
| **Flyte** | K8s-native, typed inputs/outputs, massive parallelism (10K+ map tasks), built-in caching | ML-heavy pipelines with GPU tasks |
| **Airflow** | Mature ecosystem, huge plugin library, well-understood | ETL-heavy pipelines, scheduling |
| **Prefect** | Python-native, good DX, easy local development | Smaller teams, rapid iteration |

In practice, video AI labs often use Flyte for ML workflows (it handles K8s scheduling, type safety, and large-scale parallelism natively) while keeping Airflow for traditional ETL/data warehouse jobs.

## Cleanup

```python
import shutil
shutil.rmtree(wds_dir.parent, ignore_errors=True)
shutil.rmtree(mds_dir.parent, ignore_errors=True)
```

In [ ]:
import shutil
shutil.rmtree(wds_dir.parent, ignore_errors=True)
shutil.rmtree(mds_dir.parent, ignore_errors=True)
print("Temp directories cleaned up.")

## What's Next

You now understand:
- Why specialized data formats exist (file system overhead, shuffle quality, multi-node)
- How to write and read WebDataset shards
- How to write and read MDS format
- Throughput benchmarking methodology
- How both formats handle distributed training
- How to diagnose data loading bottlenecks

**Next notebook:** [06 — System Design & Applied ML](06_system_design.ipynb) — system design exercises including end-to-end data pipeline architecture.